In [17]:
# الخطوة 1: تجهيز بيئة PySpark
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

import findspark
findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("TelecomChurn").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
print("Spark is ready!")

Spark is ready!


In [18]:
# الخطوة 2: تحميل مجموعة البيانات
import pandas as pd

# قراءة البيانات باستخدام Pandas لأخذ فكرة سريعة
data = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Shape:", data.shape)
data.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [37]:
# ── Step 1: Check missing values ─────────────────────────────────────────────
print('Missing values per column:')
print(data.isnull().sum())
print()
print('Total missing values:', data.isnull().sum().sum())

Missing values per column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Total missing values: 0


In [39]:
# ── Step 2: Fix TotalCharges ─────────────────────────────────────────────────

data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')

# Check how many NaN values appeared after conversion
print('NaN values in TotalCharges after fix:', data['TotalCharges'].isnull().sum())

# Drop those rows (only ~11 rows, safe to remove)
data.dropna(subset=['TotalCharges'], inplace=True)
data.reset_index(drop=True, inplace=True)

print('Dataset shape after dropping NaN rows:', data.shape)

NaN values in TotalCharges after fix: 11
Dataset shape after dropping NaN rows: (7032, 21)


In [40]:
data.drop(columns=['customerID'], inplace=True)
print('customerID column dropped.')

customerID column dropped.


In [41]:
# ── Step 4: Encode target column Churn (Yes → 1, No → 0) ─────────────────────
data['Churn'] = data['Churn'].map({'Yes': 1, 'No': 0})

print('Churn value counts:')
print(data['Churn'].value_counts())
print()
print('Churn rate: {:.1f}%'.format(data['Churn'].mean() * 100))

Churn value counts:
Churn
0    5163
1    1869
Name: count, dtype: int64

Churn rate: 26.6%


In [42]:
# ── Step 5: Encode categorical features ───────────────────────────────────────
# We use Label Encoding: each unique category gets a number.
from sklearn.preprocessing import LabelEncoder, StandardScaler
le = LabelEncoder()

cat_cols =data.select_dtypes(include=['object']).columns.tolist()

for col in cat_cols:
    data[col] = le.fit_transform(data[col])

print('Categorical columns encoded:', cat_cols)
print()
data.head()

Categorical columns encoded: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']



,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1


In [46]:
print('Summary Statistics for Key Numerical Features:')
data[['tenure', 'MonthlyCharges', 'TotalCharges']].describe().round(2)

Summary Statistics for Key Numerical Features:


,tenure,MonthlyCharges,TotalCharges
count,7032.00,7032.00,7032.00
mean,32.42,64.80,2283.30
std,24.55,30.09,2266.77
min,1.00,18.25,18.80
25%,9.00,35.59,401.45
50%,29.00,70.35,1397.48
75%,55.00,89.86,3794.74
max,72.00,118.75,8684.80


In [47]:
# الخطوة 3: تحميل البيانات إلى Spark DataFrame
df = spark.createDataFrame(data)
df.printSchema()
df.show(5)

root
 |-- gender: long (nullable = true)
 |-- SeniorCitizen: long (nullable = true)
 |-- Partner: long (nullable = true)
 |-- Dependents: long (nullable = true)
 |-- tenure: long (nullable = true)
 |-- PhoneService: long (nullable = true)
 |-- MultipleLines: long (nullable = true)
 |-- InternetService: long (nullable = true)
 |-- OnlineSecurity: long (nullable = true)
 |-- OnlineBackup: long (nullable = true)
 |-- DeviceProtection: long (nullable = true)
 |-- TechSupport: long (nullable = true)
 |-- StreamingTV: long (nullable = true)
 |-- StreamingMovies: long (nullable = true)
 |-- Contract: long (nullable = true)
 |-- PaperlessBilling: long (nullable = true)
 |-- PaymentMethod: long (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: long (nullable = true)

+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+-

In [48]:
from pyspark.sql.types import DoubleType

# كل الأعمدة ما عدا عمود Churn (الهدف)
feature_cols = [col for col in df.columns if col != 'Churn']

for col_name in feature_cols:
    df = df.withColumn(col_name, df[col_name].cast(DoubleType()))

# كمان نخلي عمود الهدف (Churn) double
df = df.withColumn("label", df["Churn"].cast(DoubleType()))
df = df.drop("Churn")  # هنستخدم label بداله

df.printSchema()

root
 |-- gender: double (nullable = true)
 |-- SeniorCitizen: double (nullable = true)
 |-- Partner: double (nullable = true)
 |-- Dependents: double (nullable = true)
 |-- tenure: double (nullable = true)
 |-- PhoneService: double (nullable = true)
 |-- MultipleLines: double (nullable = true)
 |-- InternetService: double (nullable = true)
 |-- OnlineSecurity: double (nullable = true)
 |-- OnlineBackup: double (nullable = true)
 |-- DeviceProtection: double (nullable = true)
 |-- TechSupport: double (nullable = true)
 |-- StreamingTV: double (nullable = true)
 |-- StreamingMovies: double (nullable = true)
 |-- Contract: double (nullable = true)
 |-- PaperlessBilling: double (nullable = true)
 |-- PaymentMethod: double (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- label: double (nullable = true)



In [49]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df = assembler.transform(df)

# هيظهر عمود جديد اسمه features
df.select("features", "label").show(5, truncate=False)

+-------------------------------------------------------------------------+-----+
|features                                                                 |label|
+-------------------------------------------------------------------------+-----+
|(19,[2,4,6,9,15,16,17,18],[1.0,1.0,1.0,2.0,1.0,2.0,29.85,29.85])         |0.0  |
|(19,[0,4,5,8,10,14,16,17,18],[1.0,34.0,1.0,2.0,2.0,1.0,3.0,56.95,1889.5])|0.0  |
|(19,[0,4,5,8,9,15,16,17,18],[1.0,2.0,1.0,2.0,2.0,1.0,3.0,53.85,108.15])  |1.0  |
|(19,[0,4,6,8,10,11,14,17,18],[1.0,45.0,1.0,2.0,2.0,2.0,1.0,42.3,1840.75])|0.0  |
|(19,[4,5,7,15,16,17,18],[2.0,1.0,1.0,1.0,2.0,70.7,151.65])               |1.0  |
+-------------------------------------------------------------------------+-----+
only showing top 5 rows



In [50]:
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)
print("Training samples:", train_data.count())
print("Test samples:", test_data.count())

Training samples: 5685
Test samples: 1347


In [51]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=20,
    maxDepth=10,
    seed=42
)

model = rf.fit(train_data)

In [52]:
predictions = model.transform(test_data)
predictions.select("label", "prediction", "probability").show(5)

+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|  0.0|       1.0|[0.40348347104240...|
|  0.0|       0.0|[0.52608557152363...|
|  1.0|       1.0|[0.45676362718766...|
|  1.0|       1.0|[0.40338721247092...|
|  0.0|       0.0|[0.63438409861545...|
+-----+----------+--------------------+
only showing top 5 rows



In [53]:
# الخطوة 6: تقييم النموذج
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print("Test Accuracy:", accuracy)

bin_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = bin_evaluator.evaluate(predictions)
print("Test AUC:", auc)

Test Accuracy: 0.7958426132145509
Test AUC: 0.8376835582717902


In [54]:
model.write().overwrite().save("churn_rf_model")

In [55]:
!zip -r churn_rf_model.zip churn_rf_model
from google.colab import files
files.download("churn_rf_model.zip")

  adding: churn_rf_model/ (stored 0%)
  adding: churn_rf_model/metadata/ (stored 0%)
  adding: churn_rf_model/metadata/._SUCCESS.crc (stored 0%)
  adding: churn_rf_model/metadata/part-00000 (deflated 47%)
  adding: churn_rf_model/metadata/_SUCCESS (stored 0%)
  adding: churn_rf_model/metadata/.part-00000.crc (stored 0%)
  adding: churn_rf_model/treesMetadata/ (stored 0%)
  adding: churn_rf_model/treesMetadata/._SUCCESS.crc (stored 0%)
  adding: churn_rf_model/treesMetadata/.part-00000-f16ade6c-a51e-49a8-b325-8d8a9a2735ec-c000.snappy.parquet.crc (stored 0%)
  adding: churn_rf_model/treesMetadata/part-00000-f16ade6c-a51e-49a8-b325-8d8a9a2735ec-c000.snappy.parquet (deflated 39%)
  adding: churn_rf_model/treesMetadata/_SUCCESS (stored 0%)
  adding: churn_rf_model/data/ (stored 0%)
  adding: churn_rf_model/data/.part-00000-e50a5d7b-8cc1-4541-9415-e3c0f5e04b16-c000.snappy.parquet.crc (stored 0%)
  adding: churn_rf_model/data/._SUCCESS.crc (stored 0%)
  adding: churn_rf_model/data/part-00000-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>